In [ ]:
import pandas as pd
import os

def scan_and_process_geography_v2():
    path = '/content/'
    files = [f for f in os.listdir(path) if f.endswith('.csv')]
    target_cols = ['zip', 'city', 'region', 'district']
    potential_mappings = {
        'zip': ['zip', 'zip_code', 'postal_code', 'zip_FK'],
        'city': ['city', 'town', 'municipality'],
        'region': ['region', 'state', 'province'],
        'district': ['district', 'county', 'area', 'suburb']
    }

    source_report = {}
    collected_dfs = []
    print("--- Bắt đầu quét các file cho bảng GEOGRAPHY (Ưu tiên dữ liệu đầy đủ) ---")
    
    for file in files:
        if file == 'geography_new.csv': continue
        try:
            file_path = os.path.join(path, file)
            temp_df = pd.read_csv(file_path, nrows=0, encoding='utf-8-sig')
            found_cols = {}
            for target, aliases in potential_mappings.items():
                for alias in aliases:
                    if alias in temp_df.columns:
                        found_cols[alias] = target
                        break

            if 'zip' in found_cols.values():
                print(f"Tìm thấy dữ liệu GEOGRAPHY trong: {file} ({list(found_cols.keys())})")
                full_df = pd.read_csv(file_path, encoding='utf-8-sig')
                mapped_df = full_df[list(found_cols.keys())].rename(columns=found_cols)
                # Thêm trọng số ưu tiên: file nào có nhiều cột mục tiêu hơn sẽ được ưu tiên lên trên
                mapped_df['priority'] = len(found_cols)
                collected_dfs.append(mapped_df)

                for alias, target in found_cols.items():
                    if target not in source_report:
                        source_report[target] = []
                    source_report[target].append(f"{file} (gốc: {alias})")
        except Exception: continue

    if not collected_dfs: 
        print("Không tìm thấy file nào chứa dữ liệu địa lý.")
        return

    # Gộp và sắp xếp để các dòng có nhiều thông tin (priority cao) nằm trên cùng
    final_df = pd.concat(collected_dfs, ignore_index=True)
    final_df = final_df.sort_values(by='priority', ascending=False)

    # Xóa trùng lặp mã ZIP, giữ lại dòng có nhiều thông tin nhất
    initial_len = len(final_df)
    final_df = final_df.drop_duplicates(subset=['zip'], keep='first')
    print(f"\nĐã xử lý: Xóa {initial_len - len(final_df)} dòng trùng lặp mã ZIP.")

    for col in target_cols:
        if col not in final_df.columns:
            final_df[col] = None

    defaults = {'zip': '00000', 'city': 'Unknown', 'region': 'Unknown', 'district': 'Unknown'}
    for col, val in defaults.items():
        final_df[col] = final_df[col].fillna(val)

    final_df[target_cols].to_csv('/content/geography_new.csv', index=False, encoding='utf-8-sig')
    
    print(f"\n--- HOÀN THÀNH ---")
    print(f"File lưu tại: /content/geography_new.csv")
    
    print("\n--- BÁO CÁO NGUỒN DỮ LIỆU (SOURCE REPORT) ---")
    for col in target_cols:
        sources = ", ".join(source_report.get(col, ["Không tìm thấy - Sử dụng mặc định"]))
        print(f"Cột '{col}': {sources}")

scan_and_process_geography_v2()